# 05 — asyncio and the Event Loop

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
import asyncio

# Provided for this lesson.
from collections.abc import Awaitable, Callable, Iterable

from lesson import Exercises, ___, run

check = Exercises('advanced/05', {1: '4ce4555fe107a71b', 2: 'acb9b1dae3f5105f', 3: 'cfc3a0999a916df6', 4: 'b6b3a7817dc8d7de', 5: '664d98a6c3990700', 6: 'cebc650415107683', 7: 'efce8be355a3edaa', 8: '986df04fc8614937', 9: 'abde2c4653d6fd68'}, practice='advanced_02_concurrency_resilience', independent='gather_in_order')

## 1. Coroutines

- `async def f()` makes a coroutine function. Calling it returns a coroutine object and runs nothing yet.
- `await` runs a coroutine and gives its result; it's only allowed inside `async def`.
- Start the loop with `asyncio.run(main())`. In these notebooks use `run(main())` from `lesson`, which also works inside Jupyter and the browser.

In [ ]:
import asyncio

async def greeting(name):
    await asyncio.sleep(0)
    return f"Hello, {name}"

coro = greeting("Ada")
print(type(coro).__name__)
print(run(coro))

**Exercise 1.** What type does calling an `async def` function return? Write its name.

In [ ]:
answer = ___
check(1, answer)

**Exercise 2.** Await the helper and return its result.

In [ ]:
import asyncio

async def double(x):
    await asyncio.sleep(0)
    return x * 2

async def main():
    return ___

check(2, run(main()))   # should be 42

**Exercise 3.** Can you use `await` inside a normal `def`? Write `True` or `False`.

In [ ]:
answer = ___
check(3, answer)

## 2. Concurrency with gather and tasks

- Awaiting one after another runs them in sequence. `asyncio.gather(a(), b())` runs them concurrently.
- `gather` returns results in the order you passed them, not the order they finished.
- `asyncio.create_task(coro)` schedules a coroutine now; await the task later for its result.

In [ ]:
import asyncio, time

async def work(label, delay):
    await asyncio.sleep(delay)
    return label

async def main():
    start = time.perf_counter()
    results = await asyncio.gather(work("slow", 0.05), work("fast", 0.01))
    return results, round(time.perf_counter() - start, 2)

print(run(main()))

**Exercise 4.** `gather(slow, fast)`: which order are the results in?

In [ ]:
predicted = ___   # e.g. ["fast", "slow"] or ["slow", "fast"]
check(4, predicted)

**Exercise 5.** Return an awaitable that starts every worker concurrently and returns results in input order. Workers in the check release each other. A sequential implementation times out.

In [ ]:
async def batch(values, worker):
    raise NotImplementedError

def observe(start):
    async def scenario():
        ready = asyncio.Event()
        started = []
        async def worker(x):
            started.append(x)
            if len(started) == 3:
                ready.set()
            await ready.wait()
            return x * x
        result = await asyncio.wait_for(start([3, 1, 2], worker), 2)
        return result, await start([], worker)
    return run(scenario())

check(5, batch, observe=observe)

**Exercise 6.** Two `sleep(0.1)` calls with `gather` take about how long? Write the seconds as a number.

In [ ]:
predicted = ___
check(6, predicted)

**Exercise 7.** Create a task, then await it.

In [ ]:
import asyncio

async def child():
    return "done"

async def main():
    task = asyncio.create_task(child())
    return ___   # await the task

check(7, run(main()))

## 3. Don't block the loop

- `time.sleep` and other blocking calls freeze every coroutine. Use `await asyncio.sleep`.
- For blocking functions you can't change, use `await asyncio.to_thread(func, *args)` (needs threads; not in the browser).
- Async helps with waiting (I/O), not with CPU-heavy work.

In [ ]:
import asyncio
print(asyncio.iscoroutinefunction(asyncio.sleep))
import time
print(asyncio.iscoroutinefunction(time.sleep))

**Exercise 8.** Inside a coroutine, which should you use to wait: `"time.sleep"` or `"asyncio.sleep"`?

In [ ]:
answer = ___
check(8, answer)

**Exercise 9.** Which `asyncio` function runs a blocking call in a thread? Write `asyncio.` plus its name.

In [ ]:
answer = "asyncio." + ___
check(9, answer)

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### Independent build — `gather_in_order`

Run the async worker concurrently on each input, in an asyncio.TaskGroup. Propagate worker failures in an exception group. Return results in input order even when completion order differs. An empty iterable gives [].

Implement the interface below, run its check, and explain one boundary case with a test of your own.

In [ ]:
async def gather_in_order(values: Iterable, worker: Callable[[object], Awaitable]) -> list:
    raise NotImplementedError

In [ ]:
check('gather_in_order')

### 2. `run_blocking`

Run a blocking function without blocking the loop.

In [ ]:
async def run_blocking(func: Callable, *args):
    """Run a blocking function without blocking the event loop (asyncio.to_thread)."""
    return ___

In [ ]:
check('run_blocking')

### 3. `first_result`

Return the first result to finish and cancel the rest.

In [ ]:
async def first_result(coroutines: list) -> object:
    """Run coroutines concurrently; return the first result to finish and cancel the rest.

    Hint: asyncio.wait(..., return_when=asyncio.FIRST_COMPLETED).
    """
    raise NotImplementedError

In [ ]:
check('first_result')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Calling an async def function returns…

   - **a)** A thread
   - **b)** Its result
   - **c)** A coroutine object that must be awaited

**2.** What does time.sleep(1) do inside a coroutine?

   - **a)** Yields to other tasks
   - **b)** Nothing
   - **c)** Blocks the whole event loop

**3.** asyncio.gather(a(), b()) runs a and b…

   - **a)** Concurrently
   - **b)** In separate processes
   - **c)** One after the other

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/05', {1: 'e64c9c28fd99', 2: '9bded35b3584', 3: '0ca6aa20463f'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [06 — TaskGroup, Cancellation, and Structured Concurrency](06_taskgroups_cancellation_timeouts.ipynb)